## Setup - Local Data Paths


In [13]:
# Google Drive mounting not needed for local execution
# Data is stored in ../data/ directory
print("Using local data directory: ../data/")

In [16]:
# Symbolic links not needed for local execution
# Data files are already in ../data/ directory
import os
print(f"Data directory exists: {os.path.exists('../data')}")
print(f"Movies file exists: {os.path.exists('../data/movies.csv')}")
print(f"Ratings file exists: {os.path.exists('../data/ratings.csv')}")

## Package Installation



In [4]:
!pip install pyspark
!pip install sentence-transformers

!apt-get update
!apt-get install openjdk-11-jdk -y

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,141 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

## Importing Necessary Packages

In [5]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DoubleType

## Setting up Spark Session

In [6]:
spark = SparkSession.builder.appName("ScaleCraftMovieRecommendation") \
  .config("spark.executor.memory", "5g") \
  .config("spark.driver.memory", "5g") \
  .config("spark.memory.offHeap.enabled", "true") \
  .config("spark.memory.offHeap.size", "5g") \
  .getOrCreate()

## Load Raw CSV Files

In [17]:
movies = spark.read.csv("../data/movies.csv", header=True, inferSchema=True)ratings_raw = spark.read.csv("../data/ratings.csv", header=True, inferSchema=True)genome_scores = spark.read.csv("../data/genome-scores.csv", header=True, inferSchema=True)

In [19]:
movies.printSchema()
ratings_raw.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



## Data Preparation - Movies and Ratings

### Cleaning and Processing Movies Data

In [21]:
movies = movies.filter(
    (F.col("movieId") >= 1) &
    (F.col("title") != "") &
    (F.col("genres") != "")
)

### Cleaning and Processing Ratings Data

In [22]:
ratings_raw = ratings_raw.filter(
    (F.col("rating") >= 0.5) & (F.col("rating") <= 5.0) &
    (F.col("userId") >= 1) &
    (F.col("movieId") >= 1) &
    (F.col("timestamp") >= 0)
  )

### Join Movies and Ratings dataframe with movieId

In [24]:
movie_ratings = movies.join(ratings_raw, on="movieId", how="inner")
movie_ratings.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- userId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



### Store Movie + Ratings into parquet file

In [40]:
movie_ratings.write.mode("overwrite").parquet("../data/movie_ratings.parquet")

## Data Preparation - Genome Vector

## Recast to necessary types

In [26]:
genome_scores = genome_scores.withColumn("movieId", F.col("movieId").cast(IntegerType())) \
                             .withColumn("tagId", F.col("tagId").cast(IntegerType())) \
                             .withColumn("relevance", F.col("relevance").cast(DoubleType()))

### Cleaning and processing Genome Scores

In [28]:
genome_scores = genome_scores.filter(
    (F.col("movieId") >= 1) &
    (F.col("tagId") >= 1) &
    ((F.col("relevance") >= 0) & (F.col("relevance") <= 1))
)

In [30]:
genome_scores.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- tagId: integer (nullable = true)
 |-- relevance: double (nullable = true)



### Calculating Relevance for each movies based on tag
Group by movieId and collect list of (tagId, relevance), aggregate tag-relevance pairs for each movie into a single list.

In [34]:
movie_relevances_df = genome_scores.groupBy("movieId").agg(
    F.collect_list(F.struct("tagId", "relevance")).alias("tag_relevances_list")
)


In [35]:
movie_relevance_vector_df = movie_relevances_df.withColumn(
    "genome_vector",
    F.transform(
        F.array_sort(F.col("tag_relevances_list"), lambda x, y: x.tagId - y.tagId),
        lambda x: x.relevance
    )
).select("movieId", "genome_vector")

In [36]:
movie_relevance_vector_df.printSchema()


root
 |-- movieId: integer (nullable = true)
 |-- genome_vector: array (nullable = false)
 |    |-- element: double (containsNull = true)



In [37]:
movie_relevance_vector_df.show(10, truncate=False)

+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Store Movie Relevance Vector into parquet file

In [39]:
movie_relevance_vector_df.write.mode("overwrite").parquet("../data/genome_vector.parquet")